In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, recall_score

In [2]:
df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
print("데이터 로드 완료. 원본 데이터 크기:", df.shape)
df.head(3)

데이터 로드 완료. 원본 데이터 크기: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [3]:
# ==========================================
# 기본 전처리 및 결측치 처리
# ==========================================
# 불필요한 고유 식별자 피처 제거
df = df.drop('customerID', axis=1)


# TotalCharges 열의 공백 결측치를 0으로 변환 후 숫자형으로 캐스팅
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [4]:
# ==========================================
# 핵심 피처 생성 (Feature Engineering)
# ==========================================

# 부가 서비스 관여도 (Service Engagement Score)
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']

df['Service_Engagement'] = 0
for service in services:
    df['Service_Engagement'] += (df[service] == 'Yes').astype(int)  # 6가지 중 1개라도 Yes 라면 Service_Engagement에 1로 들어감
#df["Service_Engagement"].describe()


# 가입 기간 범주화 (Tenure Group)
def tenure_to_group(tenure):
    if tenure <= 12:
        return 'New(0-1yr)'
    elif tenure <= 36:
        return 'Standard(1-3yr)'
    else:
        return 'Loyal(3yr+)'
df['Tenure_Group'] = df['tenure'].apply(tenure_to_group)

df.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Service_Engagement,Tenure_Group
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1,New(0-1yr)
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,One year,No,Mailed check,56.95,1889.50,No,2,Standard(1-3yr)
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,2,New(0-1yr)
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,3,Loyal(3yr+)
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0,New(0-1yr)


In [5]:
# ==========================================
# 범주형 데이터 인코딩
# ==========================================
# 이진형 데이터 변환 (Label Encoding)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# 다중 클래스 범주 데이터 변환 (One-Hot Encoding)
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
              'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
              'Contract', 'PaymentMethod', 'Tenure_Group']

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

df.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Tenure_Group_New(0-1yr),Tenure_Group_Standard(1-3yr)
0,0,0,1,0,1,0,1,29.85,29.85,0,...,False,False,False,False,False,False,True,False,True,False
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,False,False,False,True,False,False,False,True,False,True
2,1,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,False,False,False,False,True,True,False
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,False,False,False,True,False,False,False,False,False,False
4,0,0,0,0,2,1,1,70.70,151.65,1,...,False,False,False,False,False,False,True,False,True,False


# 상관관계 분석

In [6]:
# ==========================================
# 상관관계 분석
# ==========================================
corr_with_churn = df.corr()['Churn'].sort_values(ascending=False)
print("\n=== 고객 이탈(Churn)과 강한 양의 상관관계를 가진 피처 Top 5 ===")
print(corr_with_churn.drop('Churn').head(5))

print("\n=== 고객 이탈(Churn)과 강한 음의 상관관계를 가진 피처 Top 5 ===")
print(corr_with_churn.drop('Churn').tail(5))



=== 고객 이탈(Churn)과 강한 양의 상관관계를 가진 피처 Top 5 ===
Tenure_Group_New(0-1yr)           0.317580
InternetService_Fiber optic       0.308020
PaymentMethod_Electronic check    0.301919
MonthlyCharges                    0.193356
PaperlessBilling                  0.191825
Name: Churn, dtype: float64

=== 고객 이탈(Churn)과 강한 음의 상관관계를 가진 피처 Top 5 ===
TechSupport_No internet service        -0.227890
DeviceProtection_No internet service   -0.227890
StreamingTV_No internet service        -0.227890
Contract_Two year                      -0.302253
tenure                                 -0.352229
Name: Churn, dtype: float64


# 데이터 팀으로부터 전달받은 데이터셋으로 모델별 성능 & feature importance 평가

In [10]:
df = pd.read_csv('../data/telco_churn_top5_with_engineering_2.csv')
print("데이터 로드 완료. 원본 데이터 크기:", df.shape)
df.head(3)

데이터 로드 완료. 원본 데이터 크기: (7032, 11)


,tenure,Tenure_Group,MonthlyCharges,SeniorCitizen,InternetService,PaymentMethod,PaperlessBilling,TechSupport,Service_Engagement,Contract,Churn
0,1,New(0-1yr),29.85,0,DSL,Electronic check,Yes,No,1,Month-to-month,0
1,34,Standard(1-3yr),56.95,0,DSL,Mailed check,No,No,2,One year,0
2,2,New(0-1yr),53.85,0,DSL,Mailed check,Yes,No,2,Month-to-month,1


In [11]:
# ==========================================
# 범주형 데이터 인코딩
# ==========================================
# 이진형 데이터 변환 (Label Encoding)
binary_cols = ['PaperlessBilling', 'TechSupport']

le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# 다중 클래스 범주 데이터 변환 (One-Hot Encoding)
multi_cols = ['InternetService', 'TechSupport','Service_Engagement','Contract',
              'PaymentMethod', 'Tenure_Group']

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

df.head()


,tenure,MonthlyCharges,SeniorCitizen,PaperlessBilling,Churn,InternetService_Fiber optic,InternetService_No,TechSupport_1,TechSupport_2,Service_Engagement_1,...,Service_Engagement_4,Service_Engagement_5,Service_Engagement_6,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Tenure_Group_New(0-1yr),Tenure_Group_Standard(1-3yr)
0,1,29.85,0,1,0,False,False,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,34,56.95,0,0,0,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,True
2,2,53.85,0,1,1,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
3,45,42.30,0,0,0,False,False,False,True,False,...,False,False,False,True,False,False,False,False,False,False
4,2,70.70,0,1,1,True,False,False,False,False,...,False,False,False,False,False,False,True,False,True,False


# 모델1 - Random Forest

기본 모델 

In [12]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 학습/테스트 셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 예측 및 성능 평가
y_pred = rf_model.predict(X_test)
print("\n=== 예측 모델 성능 (Classification Report) ===")
print(classification_report(y_test, y_pred))

# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(5):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")




=== 예측 모델 성능 (Classification Report) ===
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1033
           1       0.56      0.48      0.52       374

    accuracy                           0.76      1407
   macro avg       0.69      0.67      0.68      1407
weighted avg       0.75      0.76      0.76      1407


=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. MonthlyCharges (0.3388)
2. tenure (0.2612)
3. InternetService_Fiber optic (0.0539)
4. Tenure_Group_New(0-1yr) (0.0415)
5. PaymentMethod_Electronic check (0.0380)


데이터 불균형 고려

In [13]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가 - class_weight = 'balanced' 추가
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 학습/테스트 셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# 예측 및 성능 평가
y_pred = rf_model.predict(X_test)
print("\n=== 예측 모델 성능 (Classification Report) ===")
print(classification_report(y_test, y_pred))

# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(7):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")




=== 예측 모델 성능 (Classification Report) ===
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1033
           1       0.54      0.47      0.51       374

    accuracy                           0.75      1407
   macro avg       0.68      0.66      0.67      1407
weighted avg       0.75      0.75      0.75      1407


=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. MonthlyCharges (0.3046)
2. tenure (0.2480)
3. InternetService_Fiber optic (0.0616)
4. Contract_Two year (0.0616)
5. Tenure_Group_New(0-1yr) (0.0416)
6. Contract_One year (0.0367)
7. PaymentMethod_Electronic check (0.0343)


class_weight로 균형을 맞추고 나니 중요 피처에서 Service Engagement와 Tenure_Group_new 가 밀리고 상관관계 분석에서 봤던 Contract_Two year (0.0469) & InternetService_Fiber optic (0.0421) 가 올라옴. 
=> 이탈에 service Engagement가 엄청 큰 영향을 주진 않는 듯? 그러나 그래도 모델 학습에 유의미한 도움은 준다.

In [27]:
# ==========================================
# 예측 모델 학습 (Random Forest) 및 평가 - class_weight balanced + 검증셋으로 threshold 조정
# ==========================================
# 피처(X)와 타겟(y) 분리
X = df.drop('Churn', axis=1)
y = df['Churn']

# 1차: train / test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2차: train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25,  # 0.25 * 0.8 = 0.2
    stratify=y_train_full,
    random_state=42
)

# 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)


# 과적합 검사
y_train_pred = rf_model.predict(X_train)
y_val_pred = rf_model.predict(X_val)
print("과적합 검사")
print("Train F1:", f1_score(y_train, y_train_pred))
print("Val F1:", f1_score(y_val, y_val_pred))



# 최적 임계값 선정
y_val_proba = rf_model.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.3, 0.7, 0.1)
best_t = 0
best_f1 = 0
for t in thresholds:
    y_pred = (y_val_proba >= t).astype(int)
    score = f1_score(y_val, y_pred)
    
    if score > best_f1:
        best_f1 = score
        best_t = t
print("\nBest threshold:", best_t)
print("Best F1:", best_f1)

y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
y_pred_custom = (y_pred_proba >= best_t).astype(int)


# 성능 지표 출력
print(f"=== [임계값 {best_t:.2f} 적용] 성능 보고서 ===")
print(classification_report(y_test, y_pred_custom))


# 결정된 threshold로 train/valid 과적합 검사 
y_train_proba = rf_model.predict_proba(X_train)[:, 1]
y_train_custom = (y_train_proba >= best_t).astype(int)

y_val_custom = (y_val_proba >= best_t).astype(int)
print("best threshold로 과적합 검사")
print("Train F1 (best_t):", f1_score(y_train, y_train_custom))
print("Val F1 (best_t):", f1_score(y_val, y_val_custom))


# 핵심 중요 피처 확인
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n=== 모델이 판단한 상위 5개 중요 예측 피처 ===")
for i in range(5):
    print(f"{i+1}. {X.columns[indices[i]]} ({importances[indices[i]]:.4f})")



과적합 검사
Train F1: 0.9946714031971581
Val F1: 0.4936708860759494

Best threshold: 0.3
Best F1: 0.5765550239234449
=== [임계값 0.30 적용] 성능 보고서 ===
              precision    recall  f1-score   support

           0       0.86      0.77      0.81      1033
           1       0.51      0.66      0.57       374

    accuracy                           0.74      1407
   macro avg       0.68      0.71      0.69      1407
weighted avg       0.77      0.74      0.75      1407

best threshold로 과적합 검사
Train F1 (best_t): 0.954855195911414
Val F1 (best_t): 0.5765550239234449

=== 모델이 판단한 상위 5개 중요 예측 피처 ===
1. MonthlyCharges (0.2869)
2. tenure (0.2482)
3. Contract_Two year (0.0615)
4. InternetService_Fiber optic (0.0569)
5. Tenure_Group_New(0-1yr) (0.0423)


# 모델 2 - XGBClassifier

In [17]:
from xgboost import XGBClassifier

In [32]:
# 데이터 분리 
X = df.drop('Churn', axis=1)
y = df['Churn']

# 1차: train / test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2차: train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25,  # 0.25 * 0.8 = 0.2
    stratify=y_train_full,
    random_state=42
)

# XGBoost 
model_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,    # 과적합 방지
    scale_pos_weight=3,  # 이탈 데이터(1)에 약 3배의 불균형 가중치 부여
    #min_child_weight=30,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model_xgb.fit(X_train, y_train)

# 과적합 검사
y_train_pred = model_xgb.predict(X_train)
y_val_pred = model_xgb.predict(X_val)
print("Train F1 :", f1_score(y_train, y_train_pred))
print("Val F1 :", f1_score(y_val, y_val_pred))



# 최적 임계값 선정
y_val_proba = model_xgb.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.3, 0.7, 0.1)
best_t = 0
best_f1 = 0
for t in thresholds:
    y_pred = (y_val_proba >= t).astype(int)
    score = recall_score(y_val, y_pred)
    
    if score > best_f1:
        best_ = score
        best_t = t
print("\nBest threshold:", best_t)
print("Best F1:", best_f1)

y_pred_proba = model_xgb.predict_proba(X_test)[:, 1]
y_pred_custom = (y_pred_proba >= best_t).astype(int)


# 성능 지표 출력
print(f"=== [임계값 {best_t:.2f} 적용] 성능 보고서 ===")
print(classification_report(y_test, y_pred_custom))

# 결정된 threshold로 train/valid 과적합 검사 
y_train_proba = model_xgb.predict_proba(X_train)[:, 1]
y_train_custom = (y_train_proba >= best_t).astype(int)

y_val_custom = (y_val_proba >= best_t).astype(int)
print("best threshold로 과적합 검사")
print("Train F1 (best_t):", f1_score(y_train, y_train_custom))
print("Val F1 (best_t):", f1_score(y_val, y_val_custom))


Train F1 : 0.6570545829042225
Val F1 : 0.6210640608034745

Best threshold: 0.6000000000000001
Best F1: 0
=== [임계값 0.60 적용] 성능 보고서 ===
              precision    recall  f1-score   support

           0       0.89      0.78      0.83      1033
           1       0.55      0.73      0.63       374

    accuracy                           0.77      1407
   macro avg       0.72      0.76      0.73      1407
weighted avg       0.80      0.77      0.78      1407

best threshold로 과적합 검사
Train F1 (best_t): 0.6751152073732719
Val F1 (best_t): 0.625


c:\Users\EZ\module_gaia\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [02:46:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## 모델 확률값으로 데이터 분석1 - 이탈 확률로 나누기

In [28]:
# 분석용 데이터프레임 생성
analysis_df = X_test.copy()

# "이탈 확률" 받아와 기존 테스트 데이터에 컬럼 추가 (X_test에 인덱스가 유지되어 있다고 가정)
analysis_df['Churn_Prob'] = y_pred_proba  # XGBoost 모델이 계산한 확률값 (0~1)
analysis_df['Actual_Churn'] = y_test # 실제 이탈 여부 (비교용)

# 2. 확률 구간(Grade)으로 그룹 나누기 (0~100%를 5개 구간으로 구분)
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
analysis_df['Prob_Grade'] = pd.cut(analysis_df['Churn_Prob'], bins=bins, labels=labels)

# 3. 그룹별 기초 통계 지표 추출
# 분석하고 싶은 주요 컬럼들을 선정합니다.
metrics = ['tenure', 'MonthlyCharges', 'Actual_Churn']

# 그룹별 평균값 계산
group_stats = analysis_df.groupby('Prob_Grade')[metrics].mean()

# 그룹별 고객 수(Count) 추가
group_stats['Customer_Count'] = analysis_df.groupby('Prob_Grade').size()

print("=== [이탈 확률 구간별 기초 통계 분석] ===")
display(group_stats)



# 4. 특정 고위험군(Very High) 고객 리스트만 따로 뽑기
high_risk_customers = analysis_df[analysis_df['Prob_Grade'] == 'Very High'].sort_values(by='Churn_Prob', ascending=False)

print(f"\n=== [초고위험군(80% 이상) 샘플 5건] ===")
display(high_risk_customers.head(5))

=== [이탈 확률 구간별 기초 통계 분석] ===


C:\Users\EZ\AppData\Local\Temp\ipykernel_30304\406401792.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_stats = analysis_df.groupby('Prob_Grade')[metrics].mean()
C:\Users\EZ\AppData\Local\Temp\ipykernel_30304\406401792.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_stats['Customer_Count'] = analysis_df.groupby('Prob_Grade').size()


,tenure,MonthlyCharges,Actual_Churn,Customer_Count
Prob_Grade,,,,
Very Low,38.238422,61.526415,0.140652,583
Low,28.211712,72.663739,0.333333,222
Medium,19.657895,75.988487,0.467105,152
High,12.322835,72.916929,0.488189,127
Very High,5.482143,72.505804,0.705357,112



=== [초고위험군(80% 이상) 샘플 5건] ===


,tenure,MonthlyCharges,SeniorCitizen,PaperlessBilling,InternetService_Fiber optic,InternetService_No,TechSupport_1,TechSupport_2,Service_Engagement_1,Service_Engagement_2,...,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Tenure_Group_New(0-1yr),Tenure_Group_Standard(1-3yr),Churn_Prob,Actual_Churn,Prob_Grade
3676,1,69.10,0,1,True,False,False,False,False,False,...,False,False,False,True,False,True,False,1.00,1,Very High
1726,1,69.60,1,1,True,False,False,False,False,False,...,False,False,False,True,False,True,False,1.00,1,Very High
2602,1,69.75,1,1,True,False,False,False,False,False,...,False,False,False,True,False,True,False,1.00,1,Very High
3899,1,78.80,0,0,True,False,False,False,True,False,...,False,False,False,True,False,True,False,0.99,1,Very High
2883,1,71.00,1,1,True,False,False,False,False,False,...,False,False,False,True,False,True,False,0.99,1,Very High


## 모델 확률값으로 데이터 분석2 - 마케팅 액션 등급별로 나누기

In [29]:
# 분석용 데이터프레임 복사 및 확률 추가
action_df = X_test.copy()
action_df['Churn_Prob'] = y_pred_proba  # 모델의 예측 확률
action_df['Actual_Churn'] = y_test.values

# 마케팅 액션 등급(Action_Grade) 생성
def assign_action_grade(row):
    prob = row['Churn_Prob']
    tenure = row['tenure']
    monthly_charges = row['MonthlyCharges']

    if prob >= 0.7:
        if monthly_charges > action_df['MonthlyCharges'].median():
            return '1_Emergency_VIP'  # 고액 결제 중인 초고위험군 (최우선 방어)
        return '2_High_Risk'           # 일반 초고위험군
    
    elif 0.4 <= prob < 0.7:
        if tenure > 24:
            return '3_Loyal_At_Risk'    # 장기 이용자 중 이탈 징후 (혜택 강화)
        return '4_Potential_Churn'      # 잠재적 이탈군
    
    else:
        return '5_Safe_Zone'            # 안정권

action_df['Action_Grade'] = action_df.apply(assign_action_grade, axis=1)

# 3. 등급별 대응 전략(Action_Strategy) 매핑
strategy_map = {
    '1_Emergency_VIP': '전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문)',
    '2_High_Risk': '단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사',
    '3_Loyal_At_Risk': '장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안',
    '4_Potential_Churn': '부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도',
    '5_Safe_Zone': '현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토'
}

action_df['Action_Strategy'] = action_df['Action_Grade'].map(strategy_map)

# 4. 결과 요약 통계 확인
action_summary = action_df.groupby('Action_Grade').agg({
    'Churn_Prob': 'mean',
    'MonthlyCharges': 'mean',
    'tenure': 'mean',
    'Action_Strategy': 'first',
    'Actual_Churn': 'count'
}).rename(columns={'Actual_Churn': 'Customer_Count'})

print("=== [고객 등급별 마케팅 액션 요약] ===")
display(action_summary.sort_index())

# 5. 실무용: 즉시 연락이 필요한 'Emergency_VIP' 명단 추출
emergency_list = action_df[action_df['Action_Grade'] == '1_Emergency_VIP'].sort_values(by='Churn_Prob', ascending=False)
print(f"\n[알림] 최우선 관리 대상(Emergency_VIP) {len(emergency_list)}명을 발견했습니다.")

=== [고객 등급별 마케팅 액션 요약] ===


,Churn_Prob,MonthlyCharges,tenure,Action_Strategy,Customer_Count
Action_Grade,,,,,
1_Emergency_VIP,0.841529,84.353409,9.568182,전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문),132
2_High_Risk,0.847792,44.503000,2.200000,단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사,50
3_Loyal_At_Risk,0.529333,89.979167,41.216667,장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안,60
4_Potential_Churn,0.542040,68.948397,9.480769,부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도,156
5_Safe_Zone,0.108800,59.985481,39.149653,현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토,1009



[알림] 최우선 관리 대상(Emergency_VIP) 132명을 발견했습니다.


# 모델 3 - GradientBoostingClassifier

In [31]:
from sklearn.ensemble import GradientBoostingClassifier

X = df.drop('Churn', axis=1)
y = df['Churn']

# 1차: train / test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2차: train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25,  # 0.25 * 0.8 = 0.2
    stratify=y_train_full,
    random_state=42
)


model_gbr = GradientBoostingClassifier(random_state=42)
model_gbr.fit(X_train, y_train)

# 과적합 검사
y_train_pred = model_gbr.predict(X_train)
y_val_pred = model_gbr.predict(X_val)
print("Train F1 :", f1_score(y_train, y_train_pred))
print("Val F1 :", f1_score(y_val, y_val_pred))


# 임계값 조정
y_val_proba = model_gbr.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.3, 0.7, 0.1)
best_t = 0
best_f1 = 0
for t in thresholds:
    y_pred = (y_val_proba >= t).astype(int)
    score = f1_score(y_val, y_pred)
    
    if score > best_f1:
        best_f1 = score
        best_t = t
print("\nBest threshold:", best_t)
print("Best F1:", best_f1)

y_pred_proba = model_gbr.predict_proba(X_test)[:, 1]
y_pred_custom = (y_pred_proba >= best_t).astype(int)

# 성능 지표 출력
print(f"\n=== [임계값 {best_t} 적용] 성능 보고서 ===")
print(classification_report(y_test, y_pred_custom))



# 결정된 threshold로 train/valid 과적합 검사 
y_train_proba = model_gbr.predict_proba(X_train)[:, 1]
y_train_custom = (y_train_proba >= best_t).astype(int)

y_val_custom = (y_val_proba >= best_t).astype(int)
print("best threshold로 과적합 검사")
print("Train F1 (best_t):", f1_score(y_train, y_train_custom))
print("Val F1 (best_t):", f1_score(y_val, y_val_custom))



Train F1 : 0.6417690417690418
Val F1 : 0.5209003215434084

Best threshold: 0.3
Best F1: 0.6241299303944315

=== [임계값 0.3 적용] 성능 보고서 ===
              precision    recall  f1-score   support

           0       0.90      0.76      0.82      1033
           1       0.54      0.77      0.63       374

    accuracy                           0.76      1407
   macro avg       0.72      0.77      0.73      1407
weighted avg       0.81      0.76      0.77      1407

best threshold로 과적합 검사
Train F1 (best_t): 0.6769456681350955
Val F1 (best_t): 0.6241299303944315


=> GradientBoost 역시 XGBoost와 비슷한 성능을 뽑아냈지만, 모델 전체적인 성능 및 기본 t = 0.5 에서의 일반화와 안정성 측면에서 조금 더 좋았던 XGBoost 선택

## 모델 확률값으로 데이터 분석 - 이탈 확률로 나누기

In [34]:
# 분석용 데이터프레임 생성
analysis_df = X_test.copy()

# "이탈 확률" 받아와 기존 테스트 데이터에 컬럼 추가 (X_test에 인덱스가 유지되어 있다고 가정)
analysis_df['Churn_Prob'] = y_pred_proba  # XGBoost 모델이 계산한 확률값 (0~1)
analysis_df['Actual_Churn'] = y_test # 실제 이탈 여부 (비교용)

# 2. 확률 구간(Grade)으로 그룹 나누기 (0~100%를 5개 구간으로 구분)
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
analysis_df['Prob_Grade'] = pd.cut(analysis_df['Churn_Prob'], bins=bins, labels=labels)

# 3. 그룹별 기초 통계 지표 추출
# 분석하고 싶은 주요 컬럼들을 선정합니다.
metrics = ['tenure', 'MonthlyCharges','Actual_Churn']

# 그룹별 평균값 계산
group_stats = analysis_df.groupby('Prob_Grade')[metrics].mean()

# 그룹별 고객 수(Count) 추가
group_stats['Customer_Count'] = analysis_df.groupby('Prob_Grade').size()

print("=== [이탈 확률 구간별 기초 통계 분석] ===")
display(group_stats)



# 4. 특정 고위험군(Very High) 고객 리스트만 따로 뽑기
high_risk_customers = analysis_df[analysis_df['Prob_Grade'] == 'Very High'].sort_values(by='Churn_Prob', ascending=False)

print(f"\n=== [초고위험군(80% 이상) 샘플 5건] ===")
display(high_risk_customers.head(5))

=== [이탈 확률 구간별 기초 통계 분석] ===


C:\Users\EZ\AppData\Local\Temp\ipykernel_30304\2089051392.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_stats = analysis_df.groupby('Prob_Grade')[metrics].mean()
C:\Users\EZ\AppData\Local\Temp\ipykernel_30304\2089051392.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_stats['Customer_Count'] = analysis_df.groupby('Prob_Grade').size()


,tenure,MonthlyCharges,Actual_Churn,Customer_Count
Prob_Grade,,,,
Very Low,48.481633,49.437857,0.044898,490
Low,38.473684,65.784928,0.119617,209
Medium,30.952607,74.153318,0.251185,211
High,18.386986,69.692808,0.455479,292
Very High,5.512195,78.388293,0.687805,205



=== [초고위험군(80% 이상) 샘플 5건] ===


,tenure,MonthlyCharges,SeniorCitizen,PaperlessBilling,InternetService_Fiber optic,InternetService_No,TechSupport_1,TechSupport_2,Service_Engagement_1,Service_Engagement_2,...,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Tenure_Group_New(0-1yr),Tenure_Group_Standard(1-3yr),Churn_Prob,Actual_Churn,Prob_Grade
3374,1,95.10,1,1,True,False,False,False,False,True,...,False,False,False,True,False,True,False,0.966143,1,Very High
2241,1,102.45,0,1,True,False,False,False,False,False,...,False,False,False,True,False,True,False,0.964329,1,Very High
4577,1,85.05,1,1,True,False,False,False,True,False,...,False,False,False,True,False,True,False,0.960972,1,Very High
1396,4,99.80,1,1,True,False,False,False,False,False,...,False,False,True,False,False,True,False,0.948166,1,Very High
3080,3,105.35,0,1,True,False,False,False,False,False,...,False,False,False,False,False,True,False,0.946855,1,Very High


## 모델 확률값으로 데이터 분석2 - 마케팅 액션 등급별로 나누기

In [35]:
# 분석용 데이터프레임 복사 및 확률 추가
action_df = X_test.copy()
action_df['Churn_Prob'] = y_pred_proba  # 모델의 예측 확률
action_df['Actual_Churn'] = y_test.values

# 마케팅 액션 등급(Action_Grade) 생성
def assign_action_grade(row):
    prob = row['Churn_Prob']
    tenure = row['tenure']
    monthly_charges = row['MonthlyCharges']

    if prob >= 0.7:
        if monthly_charges > action_df['MonthlyCharges'].median():
            return '1_Emergency_VIP'  # 고액 결제 중인 초고위험군 (최우선 방어)
        return '2_High_Risk'           # 일반 초고위험군
    
    elif 0.4 <= prob < 0.7:
        if tenure > 24:
            return '3_Loyal_At_Risk'    # 장기 이용자 중 이탈 징후 (혜택 강화)
        return '4_Potential_Churn'      # 잠재적 이탈군
    
    else:
        return '5_Safe_Zone'            # 안정권

action_df['Action_Grade'] = action_df.apply(assign_action_grade, axis=1)

# 3. 등급별 대응 전략(Action_Strategy) 매핑
strategy_map = {
    '1_Emergency_VIP': '전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문)',
    '2_High_Risk': '단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사',
    '3_Loyal_At_Risk': '장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안',
    '4_Potential_Churn': '부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도',
    '5_Safe_Zone': '현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토'
}

action_df['Action_Strategy'] = action_df['Action_Grade'].map(strategy_map)

# 4. 결과 요약 통계 확인
action_summary = action_df.groupby('Action_Grade').agg({
    'Churn_Prob': 'mean',
    'MonthlyCharges': 'mean',
    'tenure': 'mean',
    'Action_Strategy': 'first',
    'Actual_Churn': 'count'
}).rename(columns={'Actual_Churn': 'Customer_Count'})

print("=== [고객 등급별 마케팅 액션 요약] ===")
display(action_summary.sort_index())

# 5. 실무용: 즉시 연락이 필요한 'Emergency_VIP' 명단 추출
emergency_list = action_df[action_df['Action_Grade'] == '1_Emergency_VIP'].sort_values(by='Churn_Prob', ascending=False)
print(f"\n[알림] 최우선 관리 대상(Emergency_VIP) {len(emergency_list)}명을 발견했습니다.")

=== [고객 등급별 마케팅 액션 요약] ===


,Churn_Prob,MonthlyCharges,tenure,Action_Strategy,Customer_Count
Action_Grade,,,,,
1_Emergency_VIP,0.822795,86.000536,14.010714,전담 상담원 배정 및 즉시 할인 혜택 제공 (해지 방어 전문),280
2_High_Risk,0.803517,47.189205,2.352273,단기 약정 할인 쿠폰 발송 및 서비스 만족도 조사,88
3_Loyal_At_Risk,0.539579,90.475321,46.294872,장기 이용 감사 리워드 제공 및 고가 요금제 유지 혜택 제안,156
4_Potential_Churn,0.581206,52.822283,9.119565,부가 서비스 무료 체험권 제공으로 서비스 락인(Lock-in) 유도,184
5_Safe_Zone,0.141375,54.325608,45.489270,현 상태 유지 및 신규 서비스 상향 판매(Up-selling) 검토,699



[알림] 최우선 관리 대상(Emergency_VIP) 280명을 발견했습니다.


----

## 모델 최종 선정 : XGBoost !

### XGB 모델 최적 하이퍼파라미터 조합 찾기

5-fold Cross Validation

In [38]:
import os
import itertools
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# =========================================================
# 0. 설정
# =========================================================
CSV_PATH = "../data/telco_churn_top5_with_engineering_2.csv"
TARGET_COL = "Churn"
RANDOM_STATE = 42
N_SPLITS = 5

# 하이퍼파라미터 선정 기준
# 데이터 불균형 (이상 소수, 정상 다수) 인 상황에서 모델이 "이상"을 판별하는 능력을 측정하는 PR_AUC 값을 기준으로 하이퍼파라미터 튜닝 진행!
TUNING_METRIC = "pr_auc"


# =========================================================
# 1. 전처리
# =========================================================
def build_preprocessor(X: pd.DataFrame):
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", ohe, categorical_features)
        ],
        remainder="drop"
    )
    return preprocessor, numeric_features, categorical_features


# =========================================================
# 2. threshold 탐색 / 평가 함수
# =========================================================
def tune_threshold(y_true, prob, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.1, 0.91, 0.02)

    best_thr = 0.5
    best_f1 = -1.0

    for thr in thresholds:
        pred = (prob >= thr).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)

    return best_thr, best_f1


def evaluate_all_metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()

    return {
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred, zero_division=0),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "threshold": threshold,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp
    }


# =========================================================
# 3. XGB 모델 생성
# =========================================================
def build_xgb_classifier(params: dict, y_train):
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / pos if pos > 0 else 1.0

    model = XGBClassifier(
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        min_child_weight=params["min_child_weight"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight
    )
    return model


# =========================================================
# 4. 하이퍼파라미터 후보
# =========================================================
param_grid = {
    "n_estimators": [200, 400],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 4],
    "min_child_weight": [1, 3],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_alpha": [0.0, 0.1],
    "reg_lambda": [1.0, 5.0]
}

param_combinations = list(itertools.product(
    param_grid["n_estimators"],
    param_grid["learning_rate"],
    param_grid["max_depth"],
    param_grid["min_child_weight"],
    param_grid["subsample"],
    param_grid["colsample_bytree"],
    param_grid["reg_alpha"],
    param_grid["reg_lambda"]
))

print("총 조합 수:", len(param_combinations))

# =========================================================
# 5. 데이터 로드 및 split
# =========================================================
df = pd.read_csv(CSV_PATH)

X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].copy()

# 최종 test는 제일 먼저 홀드아웃
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

X_dev = X_dev.reset_index(drop=True)
y_dev = y_dev.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("dev shape:", X_dev.shape)
print("test shape:", X_test.shape)


# =========================================================
# 6. CV로 하이퍼파라미터 튜닝
# =========================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

tuning_rows = []

for idx, combo in enumerate(param_combinations, start=1):
    params = {
        "n_estimators": combo[0],
        "learning_rate": combo[1],
        "max_depth": combo[2],
        "min_child_weight": combo[3],
        "subsample": combo[4],
        "colsample_bytree": combo[5],
        "reg_alpha": combo[6],
        "reg_lambda": combo[7],
    }

    fold_scores = []
    fold_f1s = []
    fold_aucs = []
    fold_praucs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dev, y_dev), start=1):
        X_tr = X_dev.iloc[tr_idx].copy()
        y_tr = y_dev.iloc[tr_idx].copy()
        X_va = X_dev.iloc[va_idx].copy()
        y_va = y_dev.iloc[va_idx].copy()

        preprocessor, _, _ = build_preprocessor(X_tr)
        X_tr_enc = preprocessor.fit_transform(X_tr)
        X_va_enc = preprocessor.transform(X_va)

        model = build_xgb_classifier(params, y_tr)
        model.fit(X_tr_enc, y_tr)

        va_prob = model.predict_proba(X_va_enc)[:, 1]

        # fold 내부 threshold도 validation 기준으로 튜닝
        best_thr, best_f1 = tune_threshold(y_va.to_numpy(), va_prob)
        metrics = evaluate_all_metrics(y_va.to_numpy(), va_prob, threshold=best_thr)

        fold_f1s.append(metrics["f1"])
        fold_aucs.append(metrics["roc_auc"])
        fold_praucs.append(metrics["pr_auc"])

        if TUNING_METRIC == "f1":
            fold_scores.append(metrics["f1"])
        elif TUNING_METRIC == "roc_auc":
            fold_scores.append(metrics["roc_auc"])
        else:
            fold_scores.append(metrics["pr_auc"])

    row = {
        **params,
        "cv_score_mean": float(np.mean(fold_scores)),
        "cv_f1_mean": float(np.mean(fold_f1s)),
        "cv_roc_auc_mean": float(np.mean(fold_aucs)),
        "cv_pr_auc_mean": float(np.mean(fold_praucs)),
    }
    tuning_rows.append(row)

    print(f"[{idx}/{len(param_combinations)}] done | score={row['cv_score_mean']:.4f} | params={params}")

tuning_df = pd.DataFrame(tuning_rows).sort_values(
    by="cv_score_mean", ascending=False
).reset_index(drop=True)

best_params = tuning_df.iloc[0][[
    "n_estimators", "learning_rate", "max_depth", "min_child_weight",
    "subsample", "colsample_bytree", "reg_alpha", "reg_lambda"
]].to_dict()

# int로 써야 하는 값 형변환
best_params["n_estimators"] = int(best_params["n_estimators"])
best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_child_weight"] = int(best_params["min_child_weight"])

print("\n=== Best Params ===")
print(best_params)
print("\n=== Top 5 tuning results ===")
print(tuning_df.head())


# =========================================================
# 7. 선택된 하이퍼파라미터로 OOF 확률 생성 + stable threshold 찾기
# =========================================================
oof_prob = np.zeros(len(X_dev), dtype=float)
fold_thresholds = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dev, y_dev), start=1):
    X_tr = X_dev.iloc[tr_idx].copy()
    y_tr = y_dev.iloc[tr_idx].copy()
    X_va = X_dev.iloc[va_idx].copy()
    y_va = y_dev.iloc[va_idx].copy()

    preprocessor, _, _ = build_preprocessor(X_tr)
    X_tr_enc = preprocessor.fit_transform(X_tr)
    X_va_enc = preprocessor.transform(X_va)

    model = build_xgb_classifier(best_params, y_tr)
    model.fit(X_tr_enc, y_tr)

    va_prob = model.predict_proba(X_va_enc)[:, 1]
    oof_prob[va_idx] = va_prob

    best_thr, _ = tune_threshold(y_va.to_numpy(), va_prob)
    fold_thresholds.append(best_thr)

stable_threshold = float(np.median(fold_thresholds))

oof_metrics = evaluate_all_metrics(y_dev.to_numpy(), oof_prob, threshold=stable_threshold)

print("\n=== OOF Metrics ===")
print(oof_metrics)
print("fold thresholds:", fold_thresholds)
print("stable threshold (median):", stable_threshold)


# =========================================================
# 8. dev 전체 재학습 후 test 평가
# =========================================================
final_preprocessor, numeric_features, categorical_features = build_preprocessor(X_dev)
X_dev_enc = final_preprocessor.fit_transform(X_dev)
X_test_enc = final_preprocessor.transform(X_test)

final_model = build_xgb_classifier(best_params, y_dev)
final_model.fit(X_dev_enc, y_dev)

test_prob = final_model.predict_proba(X_test_enc)[:, 1]
test_metrics = evaluate_all_metrics(y_test.to_numpy(), test_prob, threshold=stable_threshold)

print("\n=== Final Test Metrics ===")
print(test_metrics)

test_pred = (test_prob >= stable_threshold).astype(int)
print("\n=== Classification Report (Test) ===")
print(classification_report(y_test, test_pred, zero_division=0))

총 조합 수: 256
dev shape: (5625, 10)
test shape: (1407, 10)
[1/256] done | score=0.6636 | params={'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0}
[2/256] done | score=0.6631 | params={'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 5.0}
[3/256] done | score=0.6637 | params={'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 1.0}
[4/256] done | score=0.6630 | params={'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 5.0}
[5/256] done | score=0.6627 | params={'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 1, 'subsample': 0.8, 